# exp_013 — Sparse-Reward Exploration Baseline Calibration (Colab)

Measures **`env_steps_to_first_reward`** (total env steps to the first positive
*extrinsic* reward = first successful trajectory) for **ICM** and **RND** across
the four ARC-AGI games × early levels, over multiple seeds. Each run **stops on the
first reward** and is **right-censored** at a per-game cap. The random-policy
baseline (the no-inductive-bias reference each method must beat) is precomputed in
`baseline_random_policy/` — e.g. ls20 L1 ≈ 50k steps, re86 L1 ≈ 2.0M, ls20 L2/L3
unreachable by random.

This notebook is **self-contained**: it clones the public repo, installs deps,
runs the sweep in parallel, aggregates the results, and lets you **download** the
per-run logs (`result.json` + `metrics.jsonl` + `config.json`) to drop back into
the repo.

> **Set Runtime ▸ Change runtime type ▸ GPU** before running.

## 1. Setup — clone repo + install deps

In [ ]:
import os

REPO_URL = "https://github.com/LavetteSinsora/ProjectArceus.git"
REPO_DIR = "/content/ProjectArceus"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 $REPO_URL $REPO_DIR
%cd $REPO_DIR
!git pull --ff-only -q || true   # pick up latest harness/baselines

# Colab already ships torch (CUDA) + numpy; install only the ARC engine + game loader,
# and the repo package itself WITHOUT deps (so torch is not churned).
!pip -q install "arc-agi>=0.9.8" "arcengine>=0.9.3"
!pip -q install -e . --no-deps

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU — set Runtime to GPU!)")

## 2. Sweep configuration

Edit `GAME_LEVELS`, `METHODS`, `SEEDS`, and `CAPS` below. The number of runs is
`len(GAME_LEVELS) × len(METHODS) × len(SEEDS)`. Caps are the censoring points and
should scale with the random baseline for that game×level (see
`baseline_random_policy/*_random_baseline.md`).

In [ ]:
from itertools import product

METHODS = ["icm", "rnd"]
SEEDS   = list(range(8))            # 8 seeds per (method × game × level)

# (game, level_index) -> hard cap (censoring point) in TOTAL env steps.
# Scaled to the precomputed uniform-random baseline E[steps]:
#   ls20 L1 ≈ 49,843 ; ls20 L2/L3 unreachable (E=∞) ; re86 L1 ≈ 2.0M (L2/L3 ≥ 1.5–3.1M).
#   tu93 / g50t: update from baseline_random_policy/ once those reports land.
CAPS = {
    ("ls20", 0): 150_000,
    ("ls20", 1): 600_000,
    ("ls20", 2): 600_000,
    ("tu93", 0): 300_000,
    ("tu93", 1): 600_000,
    ("tu93", 2): 600_000,
    ("re86", 0): 2_000_000,
    ("re86", 1): 1_000_000,
    ("re86", 2): 1_000_000,
    ("g50t", 0): 300_000,
    ("g50t", 1): 600_000,
    ("g50t", 2): 600_000,
}
DEFAULT_CAP = 300_000

# Which (game, level) cells to run. Start with all-L1 (the achievable set); expand freely.
GAME_LEVELS = [("ls20", 0), ("tu93", 0), ("re86", 0), ("g50t", 0)]
# Full 4-game × 3-level grid:  GAME_LEVELS = list(CAPS.keys())

CONCURRENCY = 2                      # parallel runs (1–2 on a single Colab GPU)

configs = [
    dict(method=m, game=g, level=l, seed=s, cap=CAPS.get((g, l), DEFAULT_CAP))
    for (g, l), m, s in product(GAME_LEVELS, METHODS, SEEDS)
]
print(f"{len(configs)} runs = {len(GAME_LEVELS)} cells × {len(METHODS)} methods × {len(SEEDS)} seeds")
for (g, l) in GAME_LEVELS:
    print(f"  {g} L{l+1}: cap {CAPS.get((g, l), DEFAULT_CAP):,}")

## 3. Run the sweep

Each run is an isolated subprocess (clean CUDA state) writing
`runs/<run_name>/result.json` + `metrics.jsonl`. Censored runs (no reward by the
cap) are the expensive ones — expect long wall-clock on the hard cells
(re86, ls20 L2/L3). You can re-run this cell; existing finished runs are kept.

In [ ]:
import subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor

LOGDIR = "/content/exp013_logs"
os.makedirs(LOGDIR, exist_ok=True)

def run_one(c):
    name = f"{c['method']}_{c['game']}_L{c['level']+1}_s{c['seed']}"
    cmd = [sys.executable, "-m",
           "JEPA.experiments.exp_013_sparse_exploration.run",
           "--method", c["method"], "--game", c["game"],
           "--level", str(c["level"]), "--seed", str(c["seed"]),
           "--max-env-steps", str(c["cap"])]
    t0 = time.time()
    with open(f"{LOGDIR}/{name}.log", "w") as f:
        rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT).returncode
    return name, rc, time.time() - t0

t0 = time.time()
with ThreadPoolExecutor(max_workers=CONCURRENCY) as ex:
    for name, rc, dt in ex.map(run_one, configs):
        print(f"[{'ok ' if rc == 0 else 'FAIL'}] {name:32s} {dt/60:6.1f} min")
print(f"\nsweep done in {(time.time() - t0) / 60:.1f} min")

## 4. Aggregate — mean ± spread of env-steps-to-first-reward

Reports, per (method × game × level): solve rate, and the median / mean / std of
`env_steps_to_first_reward` over the **solved** seeds. Censored seeds are counted
in the solve rate (they are right-censored at the cap), so compare methods on
*both* solve-rate and steps-when-solved.

In [ ]:
import glob, json
import numpy as np
from collections import defaultdict

rows = [json.load(open(f)) for f in glob.glob(
    "JEPA/experiments/exp_013_sparse_exploration/runs/*/result.json")]
agg = defaultdict(list)
for r in rows:
    agg[(r["game"], r["level_index"], r["method"])].append(r)

print(f"{'game':>5} {'lvl':>3} {'method':>6} {'n':>3} {'solved':>7} "
      f"{'median':>9} {'mean':>9} {'std':>9}")
print("-" * 60)
for k in sorted(agg):
    rs = agg[k]
    solved = np.array([r["env_steps_to_first_reward"] for r in rs if r["solved"]], float)
    med = f"{np.median(solved):,.0f}" if solved.size else "—"
    mean = f"{solved.mean():,.0f}" if solved.size else "—"
    std = f"{solved.std():,.0f}" if solved.size else "—"
    print(f"{k[0]:>5} {k[1]+1:>3} {k[2]:>6} {len(rs):>3} "
          f"{len(solved)}/{len(rs):<5} {med:>9} {mean:>9} {std:>9}")

# Save a tidy CSV next to the runs.
import csv
out = "JEPA/experiments/exp_013_sparse_exploration/calibration_results.csv"
with open(out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["method", "game", "level_index", "seed",
        "env_steps_to_first_reward", "solved", "censored", "total_env_steps"])
    w.writeheader()
    for r in rows:
        w.writerow({k: r.get(k) for k in w.fieldnames})
print(f"\nwrote {out}  ({len(rows)} runs)")

## 5. Download / commit results back to the repo

In [ ]:
import shutil

# Bundle every run's result.json + metrics.jsonl + config.json (and the CSV) into a zip.
shutil.make_archive("/content/exp013_results", "zip",
                    "JEPA/experiments/exp_013_sparse_exploration/runs")
print("zip:", os.path.getsize("/content/exp013_results.zip"), "bytes")
try:
    from google.colab import files
    files.download("/content/exp013_results.zip")
except Exception as e:
    print("Download manually from the Files pane: /content/exp013_results.zip", e)

### (optional) commit results straight back to GitHub

The `runs/` dir is git-ignored output; to push results back, use a Personal
Access Token (don't hard-code it — paste at the prompt):

```python
import getpass
tok = getpass.getpass("GitHub PAT: ")
!git config user.email "you@example.com" && git config user.name "you"
!git add -f JEPA/experiments/exp_013_sparse_exploration/runs JEPA/experiments/exp_013_sparse_exploration/calibration_results.csv
!git commit -m "exp_013 calibration results (Colab)" -q
!git push https://$tok@github.com/LavetteSinsora/ProjectArceus.git HEAD:main
```

Then `git pull` locally. Otherwise just unzip `exp013_results.zip` into
`JEPA/experiments/exp_013_sparse_exploration/runs/` in your local clone.